# Bootstrap, L=50

In [2]:
import sys
sys.path.insert(0, '../../src/')

import numpy as np
import pickle as pkl
import tensorflow as tf

from qiskit.quantum_info import Operator

from kraus_channels import KrausMap
from loss_functions import ProbabilityMSE, ProbabilityRValue
from optimization import ModelSPAM, ModelQuantumMap, Logger, model_saver

from quantum_tools import  resample
from experimental import counts_to_probs, generate_pauliInput_circuits, generate_pauli_circuits, marginalize_counts
from spam import SPAM, InitialState, CorruptionMatrix
from quantum_circuits import integrable_circuit
from tqdm.notebook import tqdm


#np.set_printoptions(threshold=sys.maxsize)
np.set_printoptions(precision=4)

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

2025-11-15 02:01:17.246319: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
def load_data(filename, n, seed, L):
    with open(filename, 'rb') as f:
        data = pkl.load(f)


    data = marginalize_counts(data, 0)

    targets = counts_to_probs(data)
    targets_spam = targets[:6**n]
    targets_map = targets[6**n:]

    np.random.seed(seed)
    
    circuit_target = integrable_circuit(n+1, L)
    unitary = Operator(circuit_target).data

    inputs_spam, _ = generate_pauliInput_circuits(n)
        
    inputs_map, circuit_list_map = (
                generate_pauli_circuits(n, None, N=5000-6**n)
        )
    
    return inputs_spam, targets_spam, inputs_map, targets_map, unitary

def fit_spam(inputs, 
             targets,
             num_iter = 3000,
             verbose = False):
    d = targets.shape[1]
    spam_model = SPAM(init = InitialState(d),
                    povm = CorruptionMatrix(d),
                    )

    spam_opt = ModelSPAM(spam_model, tf.keras.optimizers.Adam(learning_rate=0.01))
        
    spam_opt.pretrain(100, verbose=False)

    spam_opt.train(inputs = inputs,
                    targets = targets,
                    num_iter = num_iter,
                    verbose = verbose,
                )
    
    return spam_model
    

def fit_model(inputs, 
              targets, 
              spam_model,
              num_iter = 3000,
              verbose=False):
    d = targets.shape[1]
    model = ModelQuantumMap(channel = KrausMap(d = d, 
                                        rank = d**2,
                                        spam = spam_model,
                                        ),
                    loss_function = ProbabilityMSE(),
                    optimizer = tf.optimizers.Adam(learning_rate=0.01),
                    logger = Logger(loss_function_list = [ProbabilityRValue()], sample_freq=100),
                )

    model.train(inputs = inputs,
                targets = targets,
                inputs_val = [inputs],
                targets_val = [targets],
                num_iter = num_iter,
                N = 500,
                verbose=verbose
                )
    
    return model

In [4]:
path = 'data/chaos_exp_data_20251106/baseline_L=50_20251019/'
n = 4
d = 2**n
L = 50
bootstrap_samples = 10

for i in tqdm(range(8,10)):

    spam_list = []
    model_list = []

    seed = 42 + i

    inputs_spam, targets_spam, inputs_map, targets_map, unitary = load_data(path + f'seed_{seed}.pkl', n, seed, L)

    tf.random.set_seed(seed)
    for j in tqdm(range(bootstrap_samples)):
        targets_spam_bs = resample(targets_spam, 12000)
        targets_map_bs = resample(targets_map, 12000)

        spam_model = fit_spam(inputs_spam, targets_spam_bs, verbose=False)
        spam_list.append(spam_model) 


        model = fit_model(inputs_map, 
                        targets_map_bs, 
                        spam_model,
                        num_iter = 3000, 
                        verbose=False)
        model_list.append(model)


    model_saver(spam_list, f'models/integrable_spam_L=50_bootstrap_seed{seed}.model')
    model_saver(model_list, f'models/integrable_model_L=50_bootstrap_seed{seed}.model')
    

  0%|          | 0/2 [00:00<?, ?it/s]

2025-11-15 02:01:19.928065: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-11-15 02:01:19.928097: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES="-1"
2025-11-15 02:01:19.928103: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
2025-11-15 02:01:19.928106: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2025-11-15 02:01:19.928110: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: BrigidBrain
2025-11-15 02:01:19.928113: I external/local_xla/xla/stream_executor/cuda/c

  0%|          | 0/10 [00:00<?, ?it/s]

2025-11-15 02:01:23.255760: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 3981312000 exceeds 10% of free system memory.
2025-11-15 02:01:24.051939: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 11378688000 exceeds 10% of free system memory.
2025-11-15 02:05:01.821128: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 3883925504 exceeds 10% of free system memory.
2025-11-15 02:05:02.070111: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 3883925504 exceeds 10% of free system memory.
2025-11-15 02:06:03.217245: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 3883925504 exceeds 10% of free system memory.


[0.9818863327015227]
[0.981853135898987]
[0.9817069103140847]
[0.9816035777960062]
[0.9816624484477897]
[0.9818353746377512]
[0.9815112787398499]
[0.981828665669228]
[0.9816480004882544]
[0.9815961743440054]


  0%|          | 0/10 [00:00<?, ?it/s]

[0.9740469626853197]
[0.9743569132502753]
[0.9745066389704419]
[0.9741220148627231]
[0.9744375044902277]
[0.9743939514609712]
[0.9744390777587193]
[0.9743742884594614]
[0.9742946513003811]
[0.9744677963878783]
